# Johanan's Future Simulator work

In [ ]:
def hedge_portfolio_beta(current_beta, target_beta, hedge_ratio, contrat_size, size_of_position, factor_weight, index_futures_price, hedge_period, index_multiple):
    factor_weight=round(0.99/hedge_ratio,1)
    if target_beta > current_beta:
        # If target beta is greater, take a long position
        number_of_contracts = (target_beta - current_beta) * (hedge_ratio * (factor_weight * size_of_position) / contrat_size)
        hedge_profit = number_of_contracts * (index_futures_price['Adj Close'].iloc[index_futures_price.index.get_loc(o1_start_d) + hedge_period + 1] - index_futures_price['Adj Close'].iloc[index_futures_price.index.get_loc(o1_start_d) + 1])
        opt_portf_profit = B * (oos1_new_performance.loc[str(index_futures_price.index[index_futures_price.index.get_loc(o1_start_d)+ hedge_period + 1]),'Optimized Portfolio'] - oos1_new_performance.loc[str(index_futures_price.index[index_futures_price.index.get_loc(o1_start_d)+1]),'Optimized Portfolio'])
        total_profit = opt_portf_profit + hedge_profit
        action = "LONG"
    elif target_beta < current_beta:
        # If target beta is smaller, take a short position
        number_of_contracts = (current_beta - target_beta) * (hedge_ratio * (factor_weight * size_of_position) / contrat_size)
        hedge_profit = number_of_contracts * (index_futures_price['Adj Close'].iloc[index_futures_price.index.get_loc(o1_start_d) + 1] - index_futures_price['Adj Close'].iloc[index_futures_price.index.get_loc(o1_start_d) + hedge_period + 1]) * index_multiple
    
        opt_portf_profit = B * (oos1_new_performance.loc[str(index_futures_price.index[index_futures_price.index.get_loc(o1_start_d)+ hedge_period + 1]),'Optimized Portfolio'] - oos1_new_performance.loc[str(index_futures_price.index[index_futures_price.index.get_loc(o1_start_d)+1]),'Optimized Portfolio'])
    
        total_profit = opt_portf_profit + hedge_profit
    
        action = "SHORT"
    else:

        # If target beta is equal to current beta, no action needed
        action = "no action"

        number_of_contracts = 0

        total_profit = 0

    return action, number_of_contracts, hedge_profit, opt_portf_profit, total_profit, hedge_period


In [ ]:
# # MARKET FACTOR - SP_500 FUTURES MINIMUM VARIANCE HEDGE RATIO
sp500_futures = yf.download("ES=F",start=start,end=o1_end_d,interval='1mo')['Adj Close'].pct_change().dropna()
# Fama French Monthly Data using getFamaFrenchFactors module
ff3_monthly = gff.famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)
ff3_monthly.index = ff3_monthly.index.to_period('M').to_timestamp('D')
ff3_monthly = ff3_monthly.loc[sp500_futures[:o1_start_d].index]
# Regression Analysis on Market Factor Vs SP 500 Futures Data 
y = ff3_monthly['Mkt-RF']
X = sp500_futures[:o1_start_d].values.reshape(-1,1)

mkt_model = LinearRegression()
mkt_model.fit(X, y)
market_hedge_ratio = mkt_model.coef_[0]
print('Minimum variance Hedge ratio for Mkt-RF Factor: {:,.2f}'.format(market_hedge_ratio))

# Futures Contract Price for Contract Multiplier
sp500_index_futures_price = yf.download("ES=F",interval='1mo')
sp500_index_futures_price.index = pd.to_datetime(sp500_index_futures_price.index)
sp500_index_futures_price = sp500_index_futures_price.tz_localize(None)
index_multiple = 50
sp500_index_futures_price
new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-08-01')],'Open':[4396.50],'High':[4419.75],'Low':[4377.25],'Close':[4379.75],'Adj Close':[4379.75],'Volume':[1193301]})
new_row.set_index('Date',inplace=True)
new_row
sp500_index_futures_price=pd.concat([sp500_index_futures_price,new_row]).sort_index()
sp500_index_futures_price

#  B is the size of the position
size_of_market_futures_contract = index_multiple * sp500_index_futures_price['Adj Close'].iloc[sp500_index_futures_price.index.get_loc(o1_start_d) + 1]


# mkt_opt is the current beta of Market Factor
mkt_target_beta = 1.1
# Hedging the Portfolio Using Index Futures
action, number_of_contracts, hedge_profit, opt_portf_profit, total_profit, hedge_period = hedge_portfolio_beta(mkt_opt,mkt_target_beta,market_hedge_ratio, size_of_market_futures_contract, B, 1, sp500_index_futures_price, 4, index_multiple)

print("To change beta from {} to {}, {} {:.0f} contracts over the next {} months and gain/loss from {} futures position is ${:,.2f}\nGain/Loss from optimal portfolio is ${:,.2f}\nTotal Gain/Loss from all of our positions is ${:,.2f}.".format(mkt_opt, mkt_target_beta, action, number_of_contracts, hedge_period, action, hedge_profit, opt_portf_profit, total_profit))


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

Minimum variance Hedge ratio for Mkt-RF Factor: 0.96
To change beta from 1 to 1.1, LONG 1 contracts over the next 4 months and gain/loss from LONG futures position is $-84.14
Gain/Loss from optimal portfolio is $-229,018.81
Total Gain/Loss from all of our positions is $-229,102.95.


In [ ]:
# # size FACTOR - russell 2000 FUTURES MINIMUM VARIANCE HEDGE RATIO
russell_futures = yf.download("RTY=F",start=start,end=o1_end_d,interval='1mo')['Adj Close'].pct_change().dropna()
# Fama French Monthly Data using getFamaFrenchFactors module
ff3_monthly = gff.famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)
ff3_monthly.index = ff3_monthly.index.to_period('M').to_timestamp('D')
ff3_monthly = ff3_monthly.loc[russell_futures[:o1_start_d].index]
# Regression Analysis on size Factor Vs SP 500 Futures Data 
y = ff3_monthly['SMB']
X = russell_futures[:o1_start_d].values.reshape(-1,1)

smb_model = LinearRegression()
smb_model.fit(X, y)
smb_hedge_ratio = smb_model.coef_[0]
print('Minimum variance Hedge ratio for SMB Factor: {:,.2f}'.format(smb_hedge_ratio))

# Futures Contract Price for Contract Multiplier
smb_index_futures_price = yf.download("RTY=F",interval='1mo')
smb_index_futures_price.index = pd.to_datetime(smb_index_futures_price.index)
smb_index_futures_price = smb_index_futures_price.tz_localize(None)
index_multiple = 50
new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-08-01')],'Open':[2226.50],'High':[2257],'Low':[2207.5],'Close':[2210.9],'Adj Close':[2210.9],'Volume':[175613]})
new_row.set_index('Date',inplace=True)
new_row
smb_index_futures_price=pd.concat([smb_index_futures_price,new_row]).sort_index()
# new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-05-01')],'Open':[2,288.80	],'High':[2291.8],'Low':[2254.2],'Close':[2261.5],'Adj Close':[2261.5],'Volume':[204892]})
# new_row.set_index('Date',inplace=True)
# new_row
# smb_index_futures_price=pd.concat([smb_index_futures_price,new_row]).sort_index()
# smb_index_futures_price

#  B is the size of the position
size_of_smb_futures_contract = index_multiple * smb_index_futures_price['Adj Close'].iloc[smb_index_futures_price.index.get_loc(o1_start_d) + 1]


# mkt_opt is the current beta of size Factor
smb_target_beta = 0.1
# Hedging the Portfolio Using Index Futures
action, number_of_contracts, hedge_profit, opt_portf_profit, total_profit, hedge_period = hedge_portfolio_beta(smb_opt,smb_target_beta,smb_hedge_ratio, size_of_smb_futures_contract, B, 1, smb_index_futures_price, 4, index_multiple)

print("To change beta from {} to {}, {} {:.0f} contracts over the next {} months and gain/loss from {} futures position is ${:,.2f}\nGain/Loss from optimal portfolio is ${:,.2f}\nTotal Gain/Loss from all of our positions is ${:,.2f}.".format(smb_opt, smb_target_beta, action, number_of_contracts, hedge_period, action, hedge_profit, opt_portf_profit, total_profit))


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

Minimum variance Hedge ratio for SMB Factor: 0.30
To change beta from 0 to 0.1, LONG 1 contracts over the next 4 months and gain/loss from LONG futures position is $-325.97
Gain/Loss from optimal portfolio is $-229,018.81
Total Gain/Loss from all of our positions is $-229,344.79.


In [ ]:
# # value FACTOR - russell 1000 value FUTURES MINIMUM VARIANCE HEDGE RATIO
value_futures = yf.download("RTY=F",start=start,end=o1_end_d,interval='1mo')['Adj Close'].pct_change().dropna()
# Fama French Monthly Data using getFamaFrenchFactors module
ff3_monthly = gff.famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)
ff3_monthly.index = ff3_monthly.index.to_period('M').to_timestamp('D')
ff3_monthly = ff3_monthly.loc[value_futures[:o1_start_d].index]
# Regression Analysis on Market Factor Vs SP 500 Futures Data 
y = ff3_monthly['HML']
X = value_futures[:o1_start_d].values.reshape(-1,1)

val_model = LinearRegression()
val_model.fit(X, y)
hml_hedge_ratio = val_model.coef_[0]
print('Minimum variance Hedge ratio for HML Factor: {:,.2f}'.format(hml_hedge_ratio))

# Futures Contract Price for Contract Multiplier
hml_index_futures_price = yf.download("RTY=F",interval='1mo')
hml_index_futures_price.index = pd.to_datetime(hml_index_futures_price.index)
hml_index_futures_price = hml_index_futures_price.tz_localize(None)
index_multiple = 50
# new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-08-01')],'Open':[2226.50],'High':[2257],'Low':[2207.5],'Close':[2210.9],'Adj Close':[2210.9],'Volume':[175613]})
# new_row.set_index('Date',inplace=True)
# new_row
# hml_index_futures_price=pd.concat([hml_index_futures_price,new_row]).sort_index()
# new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-05-01')],'Open':[2,288.80	],'High':[2291.8],'Low':[2254.2],'Close':[2261.5],'Adj Close':[2261.5],'Volume':[204892]})
# new_row.set_index('Date',inplace=True)
# new_row
# smb_index_futures_price=pd.concat([smb_index_futures_price,new_row]).sort_index()
# smb_index_futures_price

#  B is the size of the position
size_of_value_futures_contract = index_multiple * hml_index_futures_price['Adj Close'].iloc[hml_index_futures_price.index.get_loc(o1_start_d) + 1]


# mkt_opt is the current beta of Market Factor
hml_target_beta = 0.2
# Hedging the Portfolio Using Index Futures
action, number_of_contracts, hedge_profit, opt_portf_profit, total_profit, hedge_period = hedge_portfolio_beta(hml_opt,hml_target_beta,hml_hedge_ratio, size_of_value_futures_contract, B, 1, hml_index_futures_price, 4, index_multiple)

print("To change beta from {} to {}, {} {:.0f} contracts over the next {} months and gain/loss from {} futures position is ${:,.2f}\nGain/Loss from optimal portfolio is ${:,.2f}\nTotal Gain/Loss from all of our positions is ${:,.2f}.".format(hml_opt, hml_target_beta, action, number_of_contracts, hedge_period, action, hedge_profit, opt_portf_profit, total_profit))


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

Minimum variance Hedge ratio for HML Factor: 0.01
To change beta from 0 to 0.2, LONG 3 contracts over the next 4 months and gain/loss from LONG futures position is $-649.92
Gain/Loss from optimal portfolio is $-229,018.81
Total Gain/Loss from all of our positions is $-229,668.73.


In [ ]:
# # value FACTOR - russell 1000 value FUTURES MINIMUM VARIANCE HEDGE RATIO
value_futures = yf.download("RSV=F",start=start,end=o1_end_d,interval='1mo')['Adj Close'].pct_change().dropna()
# Fama French Monthly Data using getFamaFrenchFactors module
ff3_monthly = gff.famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)
ff3_monthly.index = ff3_monthly.index.to_period('M').to_timestamp('D')
ff3_monthly = ff3_monthly.loc[value_futures[:o1_start_d].index]
# Regression Analysis on Market Factor Vs SP 500 Futures Data 
y = ff3_monthly['HML']
X = value_futures[:o1_start_d].values.reshape(-1,1)

val_model = LinearRegression()
val_model.fit(X, y)
hml_hedge_ratio = val_model.coef_[0]
print('Minimum variance Hedge ratio for HML Factor: {:,.2f}'.format(hml_hedge_ratio))

# Futures Contract Price for Contract Multiplier
hml_index_futures_price = yf.download("RSV=F",interval='1mo')
hml_index_futures_price.index = pd.to_datetime(hml_index_futures_price.index)
hml_index_futures_price = hml_index_futures_price.tz_localize(None)
index_multiple = 50
new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-08-01')],'Open':[2226.50],'High':[2257],'Low':[2207.5],'Close':[2210.9],'Adj Close':[2210.9],'Volume':[175613]})
new_row.set_index('Date',inplace=True)
new_row
hml_index_futures_price=pd.concat([hml_index_futures_price,new_row]).sort_index()
# new_row = pd.DataFrame({'Date':[pd.to_datetime('2021-05-01')],'Open':[2,288.80	],'High':[2291.8],'Low':[2254.2],'Close':[2261.5],'Adj Close':[2261.5],'Volume':[204892]})
# new_row.set_index('Date',inplace=True)
# new_row
# smb_index_futures_price=pd.concat([smb_index_futures_price,new_row]).sort_index()
# smb_index_futures_price

#  B is the size of the position
size_of_value_futures_contract = index_multiple * hml_index_futures_price['Adj Close'].iloc[hml_index_futures_price.index.get_loc(o1_start_d) + 1]


# mkt_opt is the current beta of Market Factor
hml_target_beta = 0.2
# Hedging the Portfolio Using Index Futures
action, number_of_contracts, hedge_profit, opt_portf_profit, total_profit, hedge_period = hedge_portfolio_beta(hml_opt,hml_target_beta,hml_hedge_ratio, size_of_value_futures_contract, B, 1, hml_index_futures_price, 4, index_multiple)

print("To change beta from {} to {}, {} {:.0f} contracts over the next {} months and gain/loss from {} futures position is ${:,.2f}\nGain/Loss from optimal portfolio is ${:,.2f}\nTotal Gain/Loss from all of our positions is ${:,.2f}.".format(hml_opt, hml_target_beta, action, number_of_contracts, hedge_period, action, hedge_profit, opt_portf_profit, total_profit))


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

Minimum variance Hedge ratio for HML Factor: 0.15
To change beta from 0 to 0.2, LONG 3 contracts over the next 4 months and gain/loss from LONG futures position is $-598.06
Gain/Loss from optimal portfolio is $-229,018.81
Total Gain/Loss from all of our positions is $-229,616.87.
